<a href="https://colab.research.google.com/github/CosmicVoid/Chest-X-Ray-AI-Detection/blob/main/Chest_Xray_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
#In case you don't have it installed
!pip install kagglehub

In [1]:
#Kagglehub to download the data
import kagglehub

# File Extraction & Management
import os
import zipfile
import shutil

# Pytorch Libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils as utils

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets

# Dataset

In [4]:
# Download Found Kaggle Dataset
path = kagglehub.dataset_download("jtiptj/chest-xray-pneumoniacovid19tuberculosis")

print("Path to dataset files:", path)

100%|██████████| 1.74G/1.74G [00:22<00:00, 84.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis/versions/1


In [6]:
# View Directory
print("Main Directory: ")
print(os.listdir(path))
print("Training: ")
print(os.listdir(path + "/train"))
print("Testing: ")
print(os.listdir(path + "/test"))
print("Validation: ")
print(os.listdir(path + "/val"))

Main Directory: 
['test', 'val', 'train']
Training: 
['NORMAL', 'TURBERCULOSIS', 'COVID19', 'PNEUMONIA']
Testing: 
['NORMAL', 'TURBERCULOSIS', 'COVID19', 'PNEUMONIA']
Validation: 
['NORMAL', 'TURBERCULOSIS', 'COVID19', 'PNEUMONIA']


In [2]:
# Set an Image Transformation into a format ResNet-18 can process
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Setting all images to 224 x 224
    transforms.Grayscale(num_output_channels = 3), # ResNet-18 is built for color images (3 channels), so this conversion is necessary
    transforms.ToTensor(), # Converts images into math matrices, or Tensors
    transforms.Normalize(mean = [.485, .456, .406], std = [.229, .224, .225]) # To center the dataset values around 0.0 so it matches ResNet-18's original averages to stabilize the calculations
])

In [5]:
# Create the Datasets from the folder and Applying the transformation
train_dataset = datasets.ImageFolder(root = path + '/train', transform = transform)
test_dataset = datasets.ImageFolder(root = path + '/test', transform = transform)
val_dataset = datasets.ImageFolder(root = path + '/val', transform = transform)

print(f"{len(train_dataset)} training images")
print(f"{len(test_dataset)} testing images")
print(f"{len(val_dataset)} validation images")
print(f"Class Mapping (Alphabetical Order): {train_dataset.class_to_idx}") # To check the number labels given to the 4 categories

6326 training images
771 testing images
38 validation images
Class Mapping (Alphabetical Order): {'COVID19': 0, 'NORMAL': 1, 'PNEUMONIA': 2, 'TURBERCULOSIS': 3}


In [6]:
# Create a DataLoader, which groups the data in batches (shuffling can be applied to reduce overfitting during training)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

In [7]:
# Verifying the Success of the Process
images, labels = next(iter(train_loader)) # Gets the first item (image information) from a loader and then the next item (labels)
print("Batch image shape: ", images.shape) # Info should be [32, 3, 224, 224] (batch size 32, 3 channels, and size of 224x224)
print("Batch labels: ", labels) # Should be a List of 32 random numbers between 0 and 3 (there are 4 labels) (checking shuffling)

Batch image shape:  torch.Size([32, 3, 224, 224])
Batch labels:  tensor([2, 2, 2, 2, 2, 0, 1, 3, 2, 2, 0, 1, 2, 2, 2, 2, 1, 2, 2, 2, 1, 3, 0, 2,
        3, 2, 1, 3, 3, 1, 1, 2])


# Model Architecture

In [13]:
# Load the Pre-trained ResNet-18 Model instance
model = torchvision.models.resnet18(weights = torchvision.models.ResNet18_Weights.DEFAULT) # Using Default Weights

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 217MB/s]


In [14]:
# Freeze the Core Layers first so the pre-trained knowledge doesn't get overwritten
for parameter in model.parameters():
    parameter.requires_grad = False

In [15]:
# Since ResNet-18 is built to classify 1,000 different types of objects, we need to replace the fully-connected layer (fc) with one built for our 4 outputs
model.fc = nn.Linear(model.fc.in_features, 4) # Connecting the number of input features to the 4 outputs

In [16]:
# Move the model to GPU if available/applicable for faster training
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = model.to(device)

Using device: cpu


### Note for Colab Users:

To significantly speed up training, make sure to run this notebook on a GPU runtime. You can enable a GPU by going to `Runtime` -> `Change runtime type` -> select `GPU` as the hardware accelerator.

In [17]:
# Print the model architecture to verify the modified final layer
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

# Optimization

In [18]:
# Define the Loss Function, which will evaluate the error of the prediction
loss_function = nn.CrossEntropyLoss()
# For multi-class classification, CrossEntropyLoss is appropriate since it converts the output to a probability distribution and more heavily punishes incorrect classification.

In [19]:
# Define the Optimizer which will tune the parameters to minimize loss (Adam is a popular choice that often performs well, and a common starting learning rate for Adam is 0.001.)
optimizer = optim.Adam(model.parameters(), lr = 0.001) # Pass model.parameters to tell it which tensors to manage

# Training

# Testing

# Results